# 13. Chronos-2 без обучения (Kaggle, GPU)

Предобученная модель для временных рядов `amazon/chronos-2`, без обучения на наших данных:

| модель | вход | точки |
|---|---|---|
| `chronos2` | только история ряда станции (4 недели) | все точки val и test |
| `chronos2_cov` | + ковариаты: особые дни, события у станции, осадки и снег (прогноз погоды на 24 ч) | общий подсет |

**Настройки:** Accelerator — **GPU T4 x1** (или P100), Internet — On; Input — датасет `vkr-transit-data`.
Время ~3 ч. Результат — прогнозы в Output; оценка всех моделей вместе — позже, отдельным ноутбуком.

In [ ]:
!pip install -q chronos-forecasting==2.3.2
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## Прогнозы

In [ ]:
for name, flags in [("chronos2", ""), ("chronos2_cov", "--covariates --pred-stride 7")]:
    if os.path.exists(f"{R}/preds/{name}__test.npz"):
        print(f"{name}: уже посчитан"); continue
    t = time.time()
    !cd {REPO} && python src/run_chronos.py --name {name} {flags}
    print(f"=== {name}: {(time.time() - t) / 60:.0f} мин\n", flush=True)

## Быстрая проверка качества (val)

In [ ]:
!cd {REPO} && python src/baselines.py --splits val
!cd {REPO} && python src/evaluate.py --split val --ref norm_level